# Experiment Config File Generator

In [1]:
import pm4py

import pandas as pd

## Config building blocks

In [2]:
experiments_base = '''experiments:
[experiments]
output: 
  path: [experiments_output_path]
  name: [experiments_output_name]'''

In [ ]:
experiment_base = '''- experiment_name: [experiment_name]
  seed: 42
  load:
    path: [load_path]
    case_id_col: case:concept:name
    activity_col: concept:name
    timestamp_col: time:timestamp
  split:
    type: temporal
    params:
      test_ratio: 0.2
      val_ratio: 0.0
  worktime_identification:
    bucket_len: 3600
    period_len: week
    threshold: 0.2
    threshold_ref: perc90
    week_anchor: MON
  worktime_mapping:
    worktime_anchor: 1970-01-01T00:00:00Z
    worktime_col: worktime:timestamp
  prefix:
    min_prefix_len: 2
    max_prefix_len: null
  features:
[features]
  target:
    target: [target]
    time: [time]
  transformer:
[transformer]
  model:
[model]
  output: 
    path: [experiment_output_path]
    name: [experiment_output_name]
'''

In [4]:
### Tranformers ###
# lstm/tensor transformer
lstm_transformer = '''    transformer_key: lstm_transformer
    params:
      pad_value: 0.0
      X_normalization: 0/1
      y_normalization: 0/1'''


# Last-state transformer
last_state_transformer = '''    transformer_key: last_state_transformer
    params:
      pad_value: 0.0
      X_normalization: 0/1
      y_normalization: 0/1'''

# Aggregation transformer
aggregation_transformer = '''    transformer_key: aggregation_transformer
    params:
      numeric_aggregations: [min, max, mean, std, sum]
      onehot_aggregations: [count]
      X_normalization: 0/1
      y_normalization: 0/1'''

# Index-based transformer
index_based_transformer= '''    transformer_key: index_based_transformer
    params:
      k: 3              
      pad_value: 0.0
      X_normalization: 0/1
      y_normalization: 0/1'''


### Models ###
# lstm model
lstm_model = '''    type: lstm
    params:
      num_layers: 1
      hidden_size: 64
      lr: 1e-3
      dropout: 0.2
      batch_size: 32
      epochs: 50
      loss: mae
      verbose: 1'''

# XGBoost model
xgboost_model = '''    type: xgboost
    params:
      n_estimators: 100
      max_depth: 6
      learning_rate: 0.1
      objective: reg:absoluteerror  # MAE
      random_state: 42'''

In [16]:
ct_tax_features = '''  - feature_key: raw
    params:
      source_col_name: concept:name
      encoding: onehot
      feature_name: activity_onehot
  - feature_key: time_since_last_event
    params:
      source_col_name: time:timestamp
      case_source: case:concept:name
      encoding: numeric
      unit: seconds
      feature_name: time_since_last_event_num_sec
  - feature_key: time_in_day
    params:
      source_col_name: time:timestamp
      encoding: numeric
      unit: seconds
      feature_name: time_in_day_num_sec
  - feature_key: time_in_week
    params:
      source_col_name: time:timestamp
      encoding: numeric
      unit: seconds
      feature_name: time_in_week_num_sec'''

ct_tsl_features = '''  - feature_key: raw
    params:
      source_col_name: concept:name
      encoding: onehot
      feature_name: activity_onehot
  - feature_key: time_since_last_event
    params:
      source_col_name: time:timestamp
      case_source: case:concept:name
      encoding: numeric
      unit: seconds
      feature_name: time_since_last_event_num_sec'''


wt_tsl_features = '''  - feature_key: raw
    params:
      source_col_name: concept:name
      encoding: onehot
      feature_name: activity_onehot
  - feature_key: time_since_last_event
    params:
      source_col_name: worktime:timestamp
      case_source: case:concept:name
      encoding: numeric
      unit: seconds
      feature_name: time_since_last_event_num_sec'''

In [6]:
remaining_time_target = "remaining_time"
time_until_next_event_target = "time_until_next_event"

clocktime = "clocktime"
worktime = "worktime"

## Column identification per dataset

## Config building

### Helpers

In [17]:
def build_experiment(experiment_base: str, 
                     experiment_name: str, 
                     load_path: str, 
                     features: str, 
                     target: str,
                     time: str,
                     transformer: str, 
                     model: str, 
                     experiment_output_path: str, 
                     experiment_output_name: str) -> str:
    config = experiment_base
    config = config.replace('[experiment_name]', experiment_name)
    config = config.replace('[load_path]', load_path)
    config = config.replace('[features]', features)
    config = config.replace('[target]', target)
    config = config.replace('[time]', time)
    config = config.replace('[transformer]', transformer)
    config = config.replace('[model]', model)
    config = config.replace('[experiment_output_path]', experiment_output_path)
    config = config.replace('[experiment_output_name]', experiment_output_name)
    return config

def build_experiments(experiments_base: str,
                      experiments_output_path: str, 
                      experiments_output_name: str,
                      experiments_list: list[str], ) -> str:
    config = experiments_base
    config = config.replace('[experiments_output_path]', experiments_output_path)
    config = config.replace('[experiments_output_name]', experiments_output_name)
    experiments_str = '\n'.join(experiments_list)
    config = config.replace('[experiments]', experiments_str)
    return config


### Event log specificaion

In [36]:
#########################
# CHANGE EVENT LOG HERE #
#########################
data_set = "BPI_2012_W"
load_path = f"/home/mavo/Documents/projects/data/{data_set}.xes"

### Build all LSTM configs for one event log

In [37]:
### INDIVIDUAL EXPERIMENTS ###
experiments = []

# 1) remaining time / clocktime / tax features
experiment_name = f"{data_set}_rt_ct_tax"
experiment_config = build_experiment(
    experiment_base=experiment_base,
    experiment_name=experiment_name,
    load_path=load_path,
    features=ct_tax_features,
    target=remaining_time_target,
    time=clocktime,
    transformer=lstm_transformer,
    model=lstm_model,
    experiment_output_path=f"artifacts/experiments/{data_set}",
    experiment_output_name=experiment_name
)
experiments.append(experiment_config)

# 2) remaining time / clocktime / tsl features
experiment_name = f"{data_set}_rt_ct_tsl"
experiment_config = build_experiment(
    experiment_base=experiment_base,
    experiment_name=experiment_name,
    load_path=load_path,
    features=ct_tsl_features,
    target=remaining_time_target,
    time=clocktime,
    transformer=lstm_transformer,
    model=lstm_model,
    experiment_output_path=f"artifacts/experiments/{data_set}",
    experiment_output_name=experiment_name
)
experiments.append(experiment_config)

# 3) remaining time / worktime / tsl features
experiment_name = f"{data_set}_rt_wt_tsl"
experiment_config = build_experiment(
    experiment_base=experiment_base,
    experiment_name=experiment_name,
    load_path=load_path,
    features=ct_tsl_features,
    target=remaining_time_target,
    time=worktime,
    transformer=lstm_transformer,
    model=lstm_model,
    experiment_output_path=f"artifacts/experiments/{data_set}",
    experiment_output_name=experiment_name
)
experiments.append(experiment_config)

# 4) time until next event / clocktime / tax features
experiment_name = f"{data_set}_tune_ct_tax"
experiment_config = build_experiment(
    experiment_base=experiment_base,
    experiment_name=experiment_name,
    load_path=load_path,
    features=ct_tax_features,
    target=time_until_next_event_target,
    time=clocktime,
    transformer=lstm_transformer,
    model=lstm_model,
    experiment_output_path=f"artifacts/experiments/{data_set}",
    experiment_output_name=experiment_name
)
experiments.append(experiment_config)

# 2) time until next event / clocktime / tsl features
experiment_name = f"{data_set}_tune_ct_tsl"
experiment_config = build_experiment(
    experiment_base=experiment_base,
    experiment_name=experiment_name,
    load_path=load_path,
    features=ct_tsl_features,
    target=time_until_next_event_target,
    time=clocktime,
    transformer=lstm_transformer,
    model=lstm_model,
    experiment_output_path=f"artifacts/experiments/{data_set}",
    experiment_output_name=experiment_name
)
experiments.append(experiment_config)

# 3) time until next event / worktime / tsl features
experiment_name = f"{data_set}_tune_wt_tsl"
experiment_config = build_experiment(
    experiment_base=experiment_base,
    experiment_name=experiment_name,
    load_path=load_path,
    features=ct_tsl_features,
    target=time_until_next_event_target,
    time=worktime,
    transformer=lstm_transformer,
    model=lstm_model,
    experiment_output_path=f"artifacts/experiments/{data_set}",
    experiment_output_name=experiment_name
)
experiments.append(experiment_config)

# Bundle into joint experiments
experiments_name = f"experiments_{data_set}"
experiments_config = build_experiments(
    experiments_base=experiments_base,
    experiments_list=experiments,
    experiments_output_path="artifacts/experiment_batches",
    experiments_output_name=experiments_name
)

print(experiments_config)

experiments:
- experiment_name: BPI_2012_W_rt_ct_tax
  seed: 42
  load:
    path: /home/mavo/Documents/projects/data/BPI_2012_W.xes
    case_id_col: case:concept:name
    activity_col: concept:name
    timestamp_col: time:timestamp
  split:
    type: temporal
    params:
      test_ratio: 0.2
      val_ratio: 0.0
  worktime_identification:
    bucket_len: 3600
    period_len: week
    threshold: 0.2
    threshold_ref: perc90
    week_anchor: MON
  worktime_transformation:
    worktime_anchor: 1970-01-01T00:00:00Z
    worktime_col: worktime:timestamp
  prefix:
    min_prefix_len: 2
    max_prefix_len: null
  features:
  - feature_key: raw
    params:
      source_col_name: concept:name
      encoding: onehot
      feature_name: activity_onehot
  - feature_key: time_since_last_event
    params:
      source_col_name: time:timestamp
      case_source: case:concept:name
      encoding: numeric
      unit: seconds
      feature_name: time_since_last_event_num_sec
  - feature_key: time_in_da

In [38]:
confirm = input(f"Write experiments config to {experiments_name}.yaml? [Y/n]: ") or "y"

if confirm.lower() == 'y':
    with open(f"{experiments_name}.yaml", "w") as f:
        f.write(experiments_config)
    print(f"Wrote experiments config to {experiments_name}.yaml")
else:
    print("Aborted writing experiments config.")

Wrote experiments config to experiments_BPI_2012_W.yaml


### Build all XGBoost configs for one event log

In [39]:
"""transformers = [
    ("last_state", last_state_transformer),
    ("aggregation", aggregation_transformer),
    ("index_based", index_based_transformer)
]

# Prepare experiment list
#experiments = []

# Loop over all new transformers with XGBoost
for t_name, t_cfg in transformers:
    # 0) none
    experiment_name = f"{data_set}_xgboost_{t_name}_none"
    experiment_config = build_experiment(
        experiment_base=experiment_base,
        experiment_name=experiment_name,
        load_path=load_path,
        features=tax_base_features,
        transformer=t_cfg,
        model=xgboost_model,
        experiment_output_path=f"artifacts/experiments/xgboost/{data_set}",
        experiment_output_name=experiment_name
    )
    experiments.append(experiment_config)

    # 1) coherence_resource
    experiment_name = f"{data_set}_xgboost_{t_name}_coherence_resource"
    experiment_config = build_experiment(
        experiment_base=experiment_base,
        experiment_name=experiment_name,
        load_path=load_path,
        features=tax_base_features + "\n" + coherence_resource_feature,
        transformer=t_cfg,
        model=xgboost_model,
        experiment_output_path=f"artifacts/experiments/xgboost/{data_set}",
        experiment_output_name=experiment_name
    )
    experiments.append(experiment_config)

    # 2) raw_resource
    experiment_name = f"{data_set}_xgboost_{t_name}_raw_resource"
    experiment_config = build_experiment(
        experiment_base=experiment_base,
        experiment_name=experiment_name,
        load_path=load_path,
        features=tax_base_features + "\n" + raw_resource_feature,
        transformer=t_cfg,
        model=xgboost_model,
        experiment_output_path=f"artifacts/experiments/xgboost/{data_set}",
        experiment_output_name=experiment_name
    )
    experiments.append(experiment_config)


    # if there are more relevant columns than resource
    if not df_relevant.columns.tolist() == ['org:resource']:
        # 3) coherence_all_
        experiment_name = f"{data_set}_xgboost_{t_name}_coherence_all"
        experiment_config = build_experiment(
            experiment_base=experiment_base,
            experiment_name=experiment_name,
            load_path=load_path,
            features=tax_base_features + "\n" + build_coherence_all_feature(df_relevant),
            transformer=t_cfg,
            model=xgboost_model,
            experiment_output_path=f"artifacts/experiments/xgboost/{data_set}",
            experiment_output_name=experiment_name
        )
        experiments.append(experiment_config)

        # 4) raw_all
        experiment_name = f"{data_set}_xgboost_{t_name}_raw_all"
        experiment_config = build_experiment(
            experiment_base=experiment_base,
            experiment_name=experiment_name,
            load_path=load_path,
            features=tax_base_features + "\n" + build_raw_all_feature(df_relevant),
            transformer=t_cfg,
            model=xgboost_model,
            experiment_output_path=f"artifacts/experiments/xgboost/{data_set}",
            experiment_output_name=experiment_name
        )
        experiments.append(experiment_config)

'''# Bundle into joint experiments
experiments_name = f"xgboost_{data_set}"
experiments_config = build_experiments(
    experiments_base=experiments_base,
    experiments_list=experiments,
    experiments_output_path="artifacts/experiment_batches/xgboost",
    experiments_output_name=experiments_name
)

print(experiments_config)'''"""

'transformers = [\n    ("last_state", last_state_transformer),\n    ("aggregation", aggregation_transformer),\n    ("index_based", index_based_transformer)\n]\n\n# Prepare experiment list\n#experiments = []\n\n# Loop over all new transformers with XGBoost\nfor t_name, t_cfg in transformers:\n    # 0) none\n    experiment_name = f"{data_set}_xgboost_{t_name}_none"\n    experiment_config = build_experiment(\n        experiment_base=experiment_base,\n        experiment_name=experiment_name,\n        load_path=load_path,\n        features=tax_base_features,\n        transformer=t_cfg,\n        model=xgboost_model,\n        experiment_output_path=f"artifacts/experiments/xgboost/{data_set}",\n        experiment_output_name=experiment_name\n    )\n    experiments.append(experiment_config)\n\n    # 1) coherence_resource\n    experiment_name = f"{data_set}_xgboost_{t_name}_coherence_resource"\n    experiment_config = build_experiment(\n        experiment_base=experiment_base,\n        experimen

In [40]:
# Bundle into joint experiments
experiments_name = f"experiments_{data_set}"
experiments_config = build_experiments(
    experiments_base=experiments_base,
    experiments_list=experiments,
    experiments_output_path="artifacts/experiment_batches",
    experiments_output_name=experiments_name
)

print(experiments_config)

experiments:
- experiment_name: BPI_2012_W_rt_ct_tax
  seed: 42
  load:
    path: /home/mavo/Documents/projects/data/BPI_2012_W.xes
    case_id_col: case:concept:name
    activity_col: concept:name
    timestamp_col: time:timestamp
  split:
    type: temporal
    params:
      test_ratio: 0.2
      val_ratio: 0.0
  worktime_identification:
    bucket_len: 3600
    period_len: week
    threshold: 0.2
    threshold_ref: perc90
    week_anchor: MON
  worktime_transformation:
    worktime_anchor: 1970-01-01T00:00:00Z
    worktime_col: worktime:timestamp
  prefix:
    min_prefix_len: 2
    max_prefix_len: null
  features:
  - feature_key: raw
    params:
      source_col_name: concept:name
      encoding: onehot
      feature_name: activity_onehot
  - feature_key: time_since_last_event
    params:
      source_col_name: time:timestamp
      case_source: case:concept:name
      encoding: numeric
      unit: seconds
      feature_name: time_since_last_event_num_sec
  - feature_key: time_in_da

In [41]:
"""confirm = input(f"Write experiments config to {experiments_name}.yaml? [Y/n]: ") or "y"

if confirm.lower() == 'y':
    with open(f"{experiments_name}.yaml", "w") as f:
        f.write(experiments_config)
    print(f"Wrote experiments config to {experiments_name}.yaml")
else:
    print("Aborted writing experiments config.")"""

'confirm = input(f"Write experiments config to {experiments_name}.yaml? [Y/n]: ") or "y"\n\nif confirm.lower() == \'y\':\n    with open(f"{experiments_name}.yaml", "w") as f:\n        f.write(experiments_config)\n    print(f"Wrote experiments config to {experiments_name}.yaml")\nelse:\n    print("Aborted writing experiments config.")'